In [ ]:
!pip -q install fastapi uvicorn pyngrok nest_asyncio "transformers>=4.43" "accelerate>=0.33" "bitsandbytes>=0.43"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

base_id = "Qwen/Qwen2.5-7B-Instruct"   # faster than the 12B; also good at Arabic
tok = AutoTokenizer.from_pretrained(base_id, use_fast=True)
bnb_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(base_id, quantization_config=bnb_cfg, device_map={"": 0})
model.eval()

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
from typing import Optional, List, Dict

API_TOKEN = "secret123"        # same value as in the Streamlit sidebar
app = FastAPI()

class Req(BaseModel):
    prompt: Optional[str] = None
    messages: Optional[List[Dict[str, str]]] = None
    max_length: int = 300
    temperature: float = 0.3

@app.post("/generate")
def generate(req: Req, authorization: str = Header(None)):
    if authorization != f"Bearer {API_TOKEN}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    msgs = req.messages or [{"role": "user", "content": req.prompt}]
    inputs = tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    n_in = inputs["input_ids"].shape[1]
    kw = dict(do_sample=True, temperature=req.temperature) if req.temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=req.max_length,
                             pad_token_id=tok.eos_token_id,
                             stop_strings=["Observation:"], tokenizer=tok, **kw)
    text = tok.decode(out[0][n_in:], skip_special_tokens=True)
    return {"response": text.split("Observation:")[0].strip()}

## 3) Expose with ngrok and serve (keep this cell running)

In [ ]:
import uvicorn
from pyngrok import ngrok
from kaggle_secrets import UserSecretsClient

ngrok.kill()   
ngrok.set_auth_token(UserSecretsClient().get_secret("NGROK_TOKEN"))
public_url = ngrok.connect(8000).public_url
print("PASTE THIS IN STREAMLIT:", public_url)

server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info"))
await server.serve()